# EDA: Annual Average Daily Traffic (AADT) Time Series

This notebook explores the AADT time series from bridge traffic count records before modeling.

**Goals:**
- Understand the distribution, trend, and variability of AADT over time
- Identify structural breaks (e.g. COVID-19 dip in 2020)
- Assess stationarity to inform sequence length choice for the LSTM
- Examine year-over-year growth rates

**Data:** NBI-format CSV with `year1` and `totadt1` columns. A synthetic sample is provided at `input_data/raw/sample_aadt.csv`.

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from utils.data_loader import load_adt_csv
from utils.plot_utils import plot_series

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load and Inspect Data

In [ ]:
df = load_adt_csv('../input_data/raw/sample_aadt.csv')
print(f'Shape: {df.shape}')
print(f'Year range: {df.index.min()} – {df.index.max()}')
print(f'\nSummary statistics:')
df.describe().round(0)

## 2. AADT Trend Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.index, df['totadt1'], marker='o', linewidth=1.5, markersize=4, color='steelblue')
ax.set_title('Annual Average Daily Traffic (AADT) Over Time', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('AADT (vehicles/day)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## 3. Year-over-Year Growth Rate

Understanding the growth rate helps identify structural breaks and validate that the LSTM needs to learn non-linear patterns rather than a simple trend.

In [ ]:
df['yoy_growth_pct'] = df['totadt1'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].plot(df.index, df['totadt1'], marker='o', linewidth=1.5, markersize=4, color='steelblue')
axes[0].set_ylabel('AADT')
axes[0].set_title('AADT Level and Year-over-Year Growth Rate')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
axes[0].grid(axis='y', alpha=0.4)

colors = ['tomato' if v < 0 else 'steelblue' for v in df['yoy_growth_pct'].fillna(0)]
axes[1].bar(df.index, df['yoy_growth_pct'], color=colors, alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('YoY Growth (%)')
axes[1].set_xlabel('Year')
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.show()

print(f'Mean YoY growth: {df["yoy_growth_pct"].mean():.2f}%')
print(f'Std of YoY growth: {df["yoy_growth_pct"].std():.2f}%')
print(f'Min YoY growth: {df["yoy_growth_pct"].min():.2f}% (year: {df["yoy_growth_pct"].idxmin()})')

## 4. Rolling Statistics

A 5-year rolling mean and standard deviation reveal whether the series is stationary. Non-constant mean or variance motivates MinMax scaling before LSTM training.

In [ ]:
roll = df['totadt1'].rolling(window=5)
df['roll_mean'] = roll.mean()
df['roll_std'] = roll.std()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(df.index, df['totadt1'], label='AADT', linewidth=1.2, color='steelblue')
axes[0].plot(df.index, df['roll_mean'], label='5-yr Rolling Mean', linewidth=1.5,
             linestyle='--', color='darkorange')
axes[0].set_ylabel('AADT')
axes[0].legend()
axes[0].set_title('Rolling Mean and Standard Deviation (5-year window)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

axes[1].plot(df.index, df['roll_std'], linewidth=1.2, color='tomato', label='5-yr Rolling Std')
axes[1].set_ylabel('Std Dev')
axes[1].set_xlabel('Year')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Autocorrelation

Strong autocorrelation at short lags confirms that past AADT values carry predictive signal — validating the use of a sequence-based model like LSTM.

In [ ]:
from pandas.plotting import autocorrelation_plot

lags = range(1, min(15, len(df) // 2))
acf_vals = [df['totadt1'].autocorr(lag=k) for k in lags]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(list(lags), acf_vals, color='steelblue', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(0.2, color='tomato', linewidth=1, linestyle='--', label='±0.2 threshold')
ax.axhline(-0.2, color='tomato', linewidth=1, linestyle='--')
ax.set_xlabel('Lag (years)')
ax.set_ylabel('Autocorrelation')
ax.set_title('Autocorrelation Function (ACF)')
ax.legend()
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

print('Autocorrelation by lag:')
for lag, val in zip(lags, acf_vals):
    print(f'  Lag {lag:2d}: {val:.3f}')

## 6. Key EDA Takeaways

| Observation | Implication for Modeling |
|---|---|
| Clear upward trend with non-constant mean | MinMax scaling required before LSTM training |
| Structural break in 2020 (COVID-19) | Model may underfit the recovery years; worth monitoring test residuals |
| Strong autocorrelation at lags 1–5 | Sequence length of 5–12 years is well-motivated |
| Growing variance over time | Mild heteroscedasticity; scaling helps but residuals should be inspected by decade |